In [ ]:
import pandas as pd 
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import fdrcorrection
import warnings
warnings.filterwarnings('ignore')
import subprocess
import os

from matplotlib.lines import Line2D
import matplotlib as mpl

std_chr =  [f'chr{i}' for i in list(np.arange(1,24))+['X','Y']]

## Luminal vs Basal subclass

In [3]:
def lum_bas(full):
    lum = full[full.index.isin(['KRT20','PPARG','FOXA1','GATA3','SNX31','UPK1A','UPK2','FGFR3',   ])]
    bas = full[full.index.isin(['KRT6','KRT5','COL17A1', 'CD44', 'DSC3','GSDMC','TGM1','PI3',])]
    
   # dif = pd.concat([annot_clean, ,axis= 1) 
    dif = pd.DataFrame( {'luminal': np.log2(lum.mean()),   'basal': np.log2(bas.mean())})
    dif['fc'] =dif['luminal'] - dif['basal']
    sub = full[full.index.isin(['KRT20','PPARG','FOXA1','GATA3','SNX31','UPK1A','UPK2','FGFR3',  ]+ ['KRT6','KRT5','COL17A1', 'CD44', 'DSC3','GSDMC','TGM1','PI3' ])][full.columns[full.columns.isin( dif.index )]]
    
    dif['subtype'] = 'Others'
    dif.loc[dif['fc']>= -1.4,'subtype'] = 'Luminal'
    dif.loc[dif['fc']<= -1.4,'subtype'] = 'Basal'
    dif['subtype_color'] = dif['subtype'].map({'Others':'grey','Luminal':'orange','Basal':'purple'})
    return dif

## Pre-processing

- Read raw counts data (counts.annot.tsv)
- Select protein coding genes
- Output for DESEQ analysis (pc_count.csv)

In [ ]:
### Protein coding 
ct = pd.read_csv('../data/rna_seq_data/counts_annot.tsv', sep = "\t")
pc = ct[ct['gene.type'] == 'protein_coding']
pc.index = pc['gene.symbol']
pc_clean = pc.sort_values(by = ['length'],ascending = False).drop_duplicates(subset= ['gene.symbol'])

### Length of gene for TPM 
length = pc_clean[['gene.symbol','length']]
length['kb_length'] = length['length']/1000

pc_clean = pc_clean.drop(columns = ['gene.id', 'ensembl.gene.id', 'chr', 'gene.symbol', 'gene.type','length', 'description', 'entrez.gene.id', 'uniprot.id', 'go.gene.id'])

d = pd.DataFrame(pc_clean.T.sum())
pc_clean = pc_clean[pc_clean.index.isin(d[d[0]>10].index)]
clean_col = ['Patient' + k.split('.')[0].replace('X',"") for k in pc_clean.columns]
pc_clean.columns = clean_col

qc = pd.DataFrame(np.log10(pc_clean.mean(axis = 1))).sort_values(by = [0])
qc = qc[qc[0]>2].index

pc_qc = pc_clean[pc_clean.index.isin(qc)].drop(columns = ['Patient98'])
pc_qc = pc_qc[c.ID]
pc_qc.to_csv('../data/rna_seq_data/pc_count.csv',sep = '\t')

## Figure 2A

In [ ]:
gene_subset = ['FGFR3','TP63','WNT7B',
                'CD8A','GZMA','PRF1','CXCL9','CXCL10','TBX21',
                'TAP1','TAP2','B2M','HLA-A','HLA-B','HLA-C','CD274','PDCD1LG2','CTLA4','PDCD1','LAG3','HAVCR2','TIGIT',
                'MKI67','CCNE1','BUB1','BUB1B','CCNB2','CDC25C','CDK2','MCM4','MCM6','MCM2',
                'TGFB1','TGFB2','ACTA2','COL4A1','TAGLN','SH3PXD2A',
                'BRCA2','ERCC2','FANCA','POLE','RAD51C',
                'CLDN4','CLDN3','CLDN7','VIM','TWIST1','ZEB1','ZEB2',
                'TEK','SOX17','SOX18','CDH5'
            ]

pathway_dict = {
    'FGFR3': ['FGFR3','TP63','WNT7B' ], #https://pmc.ncbi.nlm.nih.gov/articles/PMC12431237/
    'CD8+ T effector': [ 'CD8A','GZMA','PRF1','CXCL9','CXCL10','TBX21' ], 
    'APM': [ 'TAP1', 'TAP2', 'B2M', 'HLA-A', 'HLA-B', 'HLA-C' ],
    'Immune checkpoint': ['CD274', 'PDCD1LG2', 'CTLA4', 'PDCD1', 'LAG3', 'HAVCR2', 'TIGIT' ],
    'Cell cycle': ['MKI67', 'CCNE1', 'BUB1', 'BUB1B', 'CCNB2', 'CDC25C', 'CDK2', 'MCM4', 'MCM6', 'MCM2' ],
    'DDR': ['BRCA2', 'ERCC2', 'FANCA', 'POLE', 'RAD51C'],
    'TGFbeta': ['TGFB1', 'TGFB2' ],
    'F-TBRS': ['ACTA2', 'COL4A1', 'TAGLN', 'SH3PXD2A' ],
    'EMT': [ 'CLDN4', 'CLDN3', 'CLDN7', 'VIM', 'TWIST1', 'ZEB1',' ZEB2' ],
    'Angiogenesis': ['TEK', 'SOX17', 'SOX18', 'CDH5']
}

color_list = ['gold','pink','violet','orange','teal','turquoise','deepskyblue','brown','seagreen','purple']

# Group annotations
gene_dict = {}
for gene_list, group in zip( pathway_dict.values(), pathway_dict.keys()):
    for gene in gene_list:
        gene_dict[gene] = group

# Color mapping
group_palette = {}
for group, color in zip (['FGFR3','CD8+ T effector','APM','Immune checkpoint','Cell cycle','DDR','TGFbeta','F-TBRS','EMT','Angiogenesis'],
                          color_list ):
    group_palette[group] = color

# Create legend handles
handles = [
    Patch(facecolor=color, label=group)
    for group, color in group_palette.items()
]

In [5]:
pdl1 = pd.read_csv('15126_PDL1.csv',sep =',')
c1 = pd.read_csv('15_126_analysis_annotation_1022.csv',sep = ',')
c1['clinical_trial_id'] = c1['clinical_trial_id'].astype(int)
c1['ID'] = [f"Patient{int(k)}" for k in c1['clinical_trial_id']]


FileNotFoundError: [Errno 2] No such file or directory: '15126_PDL1.csv'

In [ ]:


c1_merge = c1.merge(pdl1.sort_values(by = ['TC (%)'],ascending = False).drop_duplicates(subset =['Last Name'])[['Last Name','TC (%)','IC (%)']], 
                    left_on = 'last_name',how ='left',right_on = 'Last Name')
c1_merge['TC (%)'] = c1_merge['TC (%)'].astype(float)
c1_merge['IC (%)'] = c1_merge['IC (%)'].astype(float)

c1_merge['TC'] = "Others"
c1_merge.loc[(c1_merge['TC (%)']<1),'TC'] = 'TC0'
c1_merge.loc[(c1_merge['TC (%)']<5)*(c1_merge['TC (%)']>=1),'TC'] = 'TC1'
c1_merge.loc[(c1_merge['TC (%)']>=5),'TC'] = 'TC2'

c1_merge['IC'] = "Others"
c1_merge.loc[(c1_merge['IC (%)']<1),'IC'] = 'IC0'
c1_merge.loc[(c1_merge['IC (%)']<5)*(c1_merge['IC (%)']>=1),'IC'] = 'IC1'
c1_merge.loc[(c1_merge['IC (%)']>=5),'IC'] = 'IC2'

c1_merge.index = c1_merge.ID

In [ ]:
m15126 = (pd.read_csv('../data/rna_seq_data/deseq_result_pc_full.csv', sep = ','))
m15126_full = (pd.read_csv('../data/rna_seq_data/dds_pc_normalize_full.csv', sep = ','))

m15126_full.index = m15126['ID']
m15126_dif = lum_bas(m15126_full)
sub = m15126_full[m15126_full.index.isin(['KRT20','PPARG','FOXA1','GATA3','SNX31','UPK1A','UPK2','FGFR3',  ]+ ['KRT6','KRT5','COL17A1', 'CD44', 'DSC3','GSDMC','TGM1','PI3' ])][m15126_full.columns[m15126_full.columns.isin( m15126_dif.index )]]


: 

In [ ]:
## clinical response + PD L1




: 

In [ ]:
from statsmodels.stats.multitest import fdrcorrection
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt 
import seaborn as sns

from scipy.stats import mannwhitneyu 
from statsmodels.stats.multitest import fdrcorrection

sns.set_style("whitegrid", {'axes.grid' : False})

: 

: 

: 

: 